# Installing Deps

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import tiktoken
import sentencepiece as spm
import numpy as np
from torch.utils.data import DataLoader
import tqdm
import math

# Tokenizer

## Loading data

In [ ]:
!wget https://raw.githubusercontent.com/karpathy/ng-video-lecture/refs/heads/master/input.txt

--2025-10-28 05:32:11--  https://raw.githubusercontent.com/karpathy/ng-video-lecture/refs/heads/master/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.1’

input.txt.1         100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2025-10-28 05:32:11 (35.2 MB/s) - ‘input.txt.1’ saved [1115394/1115394]



In [ ]:
with open('input.txt', 'r', encoding='utf-8') as f:
  data=f.read()
n=len(data)
train_data=data[:int(n*0.9)]
val_data=data[int(n*0.9):]

## Training Sentencepiece Tokenizer

In [ ]:
spm.SentencePieceTrainer.train(
    input='input.txt',
    model_prefix='tokenizer',
    vocab_size=33488,
    model_type='bpe',
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3,
    pad_piece='[PAD]',
    unk_piece='[UNK]',
    bos_piece='[BOS]',
    eos_piece='[EOS]',
    user_defined_symbols=['[CLS]', '[SEP]', '[MASK]']

)

In [ ]:
sp=spm.SentencePieceProcessor(model_file='tokenizer.model')
encoded_pieces=sp.piece_to_id('[MASK]')
print(encoded_pieces)

6


In [ ]:
print(sp.get_piece_size())

33488


In [ ]:
print(sp.decode(encoded_pieces))

[MASK]


## Custom dataset Class

# ModelArgs

In [ ]:
class ModelArgs:
  base_freq=10000.0
  batch_size=32
  embed_dims=512
  max_seq_len=256
  num_heads=8
  attn_dropout=0.2
  eps=1e-6
  device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
  num_blocks=8
  dropout=0.2
  vocab_size=sp.get_piece_size()
  forward_dim=6
  forward_eps=1e-3
  AdamW_weight_decay=1e-1
  decay_lr=True
  beta1=0.9
  beta2=0.95



In [ ]:
ModelArgs.vocab_size

33488

# RoPE

In [ ]:
class RoPE(nn.Module):
  def __init__(self, base_freq=ModelArgs.base_freq, embed_dims=ModelArgs.embed_dims, max_seq_len=ModelArgs.max_seq_len):
    super().__init__()
    self.base_freq=base_freq
    self.embed_dims=embed_dims
    self.max_seq_len=max_seq_len
    positions=torch.arange(0, self.max_seq_len, dtype=torch.float)
    theta_num=torch.arange(0,self.embed_dims, 2).float()
    theta=1.0/(self.base_freq**(theta_num/self.embed_dims))
    self.register_buffer("theta", theta)
    angles=positions.unsqueeze(1)*theta.unsqueeze(0)
    self.register_buffer("sine", torch.sin(angles))
    self.register_buffer("cosine", torch.cos(angles))

  def forward(self, x, start_pos=0):
    b,s,h,d=x.shape
    x=x.view(b,s,h,d//2,2)
    cos=self.cosine[start_pos:start_pos+s].unsqueeze(0).unsqueeze(2)
    sin=self.sine[start_pos:start_pos+s].unsqueeze(0).unsqueeze(2)

    x_rot=torch.stack([
        x[...,0]*cos-x[...,1]*sin,
        x[...,0]*sin+x[...,1]*cos
    ], dim=-1)
    return x_rot.view(b,s,h,d)


# RMS Norm

In [ ]:
class RMSNorm(nn.Module):
  def __init__(self, embed_dims=ModelArgs.embed_dims, eps=ModelArgs.eps, device=None):
    super().__init__()
    self.eps=eps
    self.embed_dims=embed_dims
    self.gemma=nn.Parameter(torch.ones(self.embed_dims, device=device))

  def norm(self, x):
    val=x.pow(2).mean(-1, keepdim=True)
    val=torch.sqrt(val+self.eps)
    return x/val

  def forward(self, x):
    return self.gemma*self.norm(x)

# MHA (Without CausalAttention)

In [ ]:
class MultiHeadBiAttention(nn.Module):
  def __init__(self, embed_dims=ModelArgs.embed_dims, num_heads=ModelArgs.num_heads, attn_dropout=ModelArgs.attn_dropout):
    super().__init__()
    self.embed_dims=embed_dims
    self.num_heads=num_heads
    assert self.embed_dims%self.num_heads==0, "embed_dims must be divisible by num_heads"
    self.attn_dropout=attn_dropout
    self.head_dims=embed_dims//self.num_heads
    self.qk_norm=RMSNorm(self.head_dims)

    self.Wq=nn.Linear(embed_dims, self.head_dims*self.num_heads, bias=False)
    self.Wk=nn.Linear(embed_dims, self.head_dims*self.num_heads, bias=False)
    self.Wv=nn.Linear(embed_dims, self.head_dims*self.num_heads, bias=False)
    self.out=nn.Linear(self.head_dims*self.num_heads, self.embed_dims, bias=False)
    self.rope=RoPE(embed_dims=self.head_dims)
    self.dropout=nn.Dropout(self.attn_dropout)

  def forward(self, x, qk_norm=False):
    B,S,D=x.shape
    q=self.Wq(x)
    k=self.Wk(x)
    v=self.Wv(x)

    if qk_norm:
      q=self.qk_norm(q)
      k=self.qk_norm(k)

    q=q.view(B,S,self.num_heads, self.head_dims)
    k=k.view(B,S,self.num_heads, self.head_dims)
    v=v.view(B,S,self.num_heads, self.head_dims)

    q=self.rope(q)
    k=self.rope(k)

    q=q.transpose(1,2) #(B,S,N,d)-->(B,N,S,d)
    k=k.transpose(1,2)
    v=v.transpose(1,2)

    ## attn
    attn=torch.matmul(q, k.transpose(-2,-1))/(torch.sqrt(torch.tensor(self.head_dims))) # (B,N,S,d) @ (B,N,d,S)--> (B,N,S,S)
    attn_output=F.softmax(attn,dim=-1)
    attn_output=self.dropout(attn_output)
    attn_output=torch.matmul(attn_output, v) # (B,N,S,S) @ (B,N,S,d) --> (B,N,S,d)
    attn_output=attn_output.transpose(1,2) # (B,N,S,d) --> (B,S,N,d)
    attn_output=attn_output.contiguous().view(B,S,self.num_heads*self.head_dims)
    attn_output=self.out(attn_output)
    return attn_output




# MLP

## Swiglu

In [ ]:
class Swish(nn.Module):
  def __init__(self):
    super().__init__()
    self.sigmoid=nn.Sigmoid()
  def forward(self, x):
    return x*self.sigmoid(x)


In [ ]:
class MLP(nn.Module):
  def __init__(self, embed_dims=ModelArgs.embed_dims, device=None):
    super().__init__() # Added this line
    self.embedding_dims=embed_dims
    self.batch_size=ModelArgs.batch_size
    self.hidden_dims=((self.embedding_dims*2)*4)//3
    self.linear1=nn.Linear(self.embedding_dims, self.hidden_dims, bias=False, device=device)
    self.linear2=nn.Linear(self.embedding_dims, self.hidden_dims, bias=False, device=device)
    self.linear3=nn.Linear(self.hidden_dims, self.embedding_dims, bias=False, device=device)
    self.swish=Swish()

  def forward(self, x):
    x1=self.linear1(x)
    x2=self.linear2(x)
    hidden=torch.mul(x1, self.swish(x2))
    return self.linear3(hidden)

# Encoder_Block

In [ ]:
class Encoder_layer(nn.Module):
  def __init__(self, embed_dims=ModelArgs.embed_dims, num_heads=ModelArgs.num_heads, attn_dropout=ModelArgs.attn_dropout,dropout=ModelArgs.dropout, device=None):
    super().__init__()
    self.embed_dims=embed_dims
    self.num_heads=num_heads
    self.attn_dropout=attn_dropout
    self.dropout=dropout
    self.attn=MultiHeadBiAttention(embed_dims=self.embed_dims, num_heads=self.num_heads, attn_dropout=self.attn_dropout)
    self.mlp=MLP(embed_dims=self.embed_dims, device=device)
    self.norm1=RMSNorm(embed_dims=self.embed_dims, device=device)
    self.norm2=RMSNorm(embed_dims=self.embed_dims, device=device)
    self.dropout_1=nn.Dropout(self.dropout) #Remove weights in code 1
    self.dropout_2=nn.Dropout(self.dropout) # Remove weights in code 2
  def forward(self, x):
    b,s,d=x.shape
    x=self.norm1(x)
    x=x+self.dropout_1(self.attn(x))
    x=self.norm2(x)
    x=x+self.dropout_2(self.mlp(x))
    return x


# Block

In [ ]:
class LlaDA(nn.Module):
  def __init__(self, embed_dims=ModelArgs.embed_dims, num_heads=ModelArgs.num_heads, attn_dropout=ModelArgs.attn_dropout, num_blocks=ModelArgs.num_blocks,  device=None):
    super().__init__()
    self.embed_dims=embed_dims
    self.num_heads=num_heads
    self.attn_dropout=attn_dropout
    self.num_blocks=num_blocks
    self.device=device
    self.embeddings=nn.Embedding(ModelArgs.vocab_size, self.embed_dims)
    self.layers=nn.ModuleList([Encoder_layer(embed_dims=self.embed_dims, num_heads=self.num_heads, attn_dropout=self.attn_dropout, device=self.device) for _ in range(self.num_blocks)])
    self.norm=RMSNorm(embed_dims=self.embed_dims, device=self.device)
    self.dropout=nn.Dropout(ModelArgs.dropout)
    self.output_layer=nn.Linear(self.embed_dims, ModelArgs.vocab_size, bias=False)

  def forward(self, x):
    b,s=x.shape
    x=self.embeddings(x)
    for layer in self.layers:
      x=layer(x)
    x=self.norm(x)
    x=self.dropout(x)
    x=self.output_layer(x)
    return x



# MAIN PART: Training this model

## Creating dataset

In [ ]:
class Dataset(torch.utils.data.Dataset):
  def __init__(self, text_file_path, tokenizer_path, ctx_len=ModelArgs.max_seq_len, split="train",val_split=0.1):
    self.text_file=text_file_path
    self.ctx_len=ctx_len
    self.sp=spm.SentencePieceProcessor(model_file=tokenizer_path)
    self.split=split
    self.val_split=val_split
    with open(self.text_file, 'r', encoding='utf-8') as f:
      self.data=f.read()
    all_tokens=self.sp.encode_as_ids(self.data)
    n=len(all_tokens)
    train_len=int(n*(1-self.val_split))
    if self.split=="train":
      self.data_np=np.array(all_tokens[:train_len], dtype=np.int64)
    else:
      self.data_np=np.array(all_tokens[train_len:],dtype=np.int64)

    self.data=torch.from_numpy(self.data_np)

    self._n=max(0, len(self.data)-self.ctx_len)
  def __len__(self):
    return self._n

  def __getitem__(self, idx):
    return self.data[idx:idx+self.ctx_len]



In [ ]:
text_file_path='input.txt'
tokenizer_path='tokenizer.model'
train_dataset=Dataset(text_file_path, tokenizer_path, split="train")
val_dataset=Dataset(text_file_path, tokenizer_path, split="val")

In [ ]:
sample_tensor=train_dataset[0]

In [ ]:
print(sample_tensor)

tensor([  427,   811, 33452,  2092,    88,  2442,   552,  2018, 33444,   428,
           72,   366, 33453,   946, 33452,  2087, 33444,   366, 33453,   427,
          811, 33452,   323,   186,   161,  3726,  1238,    39,   716,   285,
           39,  7281, 33472,   946, 33452, 11330, 33453,  3726, 33453,   427,
          811, 33452,   427, 33444,    40,   263,  3188,  1107,    80,  3468,
         1686,    39,    17,   937, 33453,   946, 33452,   397,   263, 33458,
        33431, 33444,    88,   263, 33458, 33431, 33453,   427,   811, 33452,
          637,   289,  1062,   117, 33444,    49,    88, 33458,    25,   116,
         2798,   222,   182,   590,  5291, 33453,   442, 33458, 33431,     9,
        11286, 33472,   946, 33452,   443,   243,  6576,   130, 33458, 33431,
        33466,   296,    95,    57,   593, 33452,   708, 33444,   708, 33474,
          675,   811, 33452,  1588,   461, 33444,   208,  2752, 33453,   427,
          811, 33452,   397,   186, 11597,   705,  2752, 33444, 

In [ ]:
train_dataloader=DataLoader(
    train_dataset,
    batch_size=ModelArgs.batch_size,
    shuffle=True
)
val_dataloader=DataLoader(
    val_dataset,
    batch_size=ModelArgs.batch_size,
)

In [ ]:
len(train_dataloader)

7379

## Model Initialisation

In [ ]:
model=LlaDA(device=ModelArgs.device)
model.to(ModelArgs.device)

LlaDA(
  (embeddings): Embedding(33488, 512)
  (layers): ModuleList(
    (0-7): 8 x Encoder_layer(
      (attn): MultiHeadBiAttention(
        (qk_norm): RMSNorm()
        (Wq): Linear(in_features=512, out_features=512, bias=False)
        (Wk): Linear(in_features=512, out_features=512, bias=False)
        (Wv): Linear(in_features=512, out_features=512, bias=False)
        (out): Linear(in_features=512, out_features=512, bias=False)
        (rope): RoPE()
        (dropout): Dropout(p=0.2, inplace=False)
      )
      (mlp): MLP(
        (linear1): Linear(in_features=512, out_features=1365, bias=False)
        (linear2): Linear(in_features=512, out_features=1365, bias=False)
        (linear3): Linear(in_features=1365, out_features=512, bias=False)
        (swish): Swish(
          (sigmoid): Sigmoid()
        )
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
      (dropout_1): Dropout(p=0.2, inplace=False)
      (dropout_2): Dropout(p=0.2, inplace=False)
    )
  )
  (norm): RM

## Forward process

In [ ]:
import time

In [ ]:
def forward_process(batch, total_dim=ModelArgs.forward_dim, eps=ModelArgs.forward_eps):
  b,l=batch.shape
  t=torch.rand((b,), device=batch.device)
  p_mask=(1-eps)*t+eps
  p_mask=p_mask[:, None].repeat(1,l)
  mask_indices=torch.rand((b,l), device=batch.device)<p_mask
  noisy_batch=torch.where(mask_indices, total_dim, batch)
  return noisy_batch, mask_indices, p_mask

In [ ]:
batch=next(iter(train_dataloader))
batch=batch.to(ModelArgs.device)
noisy_batch, mask_indices, p_mask=forward_process(batch)

In [ ]:
print(noisy_batch)


tensor([[    6, 33444,  1029,  ...,   886,     6,  1284],
        [    6,  1439,    49,  ...,     6,     6,     6],
        [ 4993, 33466,  1523,  ...,   127,    61,   502],
        ...,
        [    6,    39,     6,  ...,     6,     6,     6],
        [  147,     6,   716,  ...,     6,   166,     6],
        [ 1258,    61,     6,  ...,   179,   183,    17]], device='cuda:0')


In [ ]:
print(mask_indices)

tensor([[ True, False, False,  ..., False,  True, False],
        [ True, False, False,  ...,  True,  True,  True],
        [False, False, False,  ..., False, False, False],
        ...,
        [ True, False,  True,  ...,  True,  True,  True],
        [False,  True, False,  ...,  True, False,  True],
        [False, False,  True,  ..., False, False, False]], device='cuda:0')


In [ ]:
p_mask

tensor([[0.4118, 0.4118, 0.4118,  ..., 0.4118, 0.4118, 0.4118],
        [0.7327, 0.7327, 0.7327,  ..., 0.7327, 0.7327, 0.7327],
        [0.2066, 0.2066, 0.2066,  ..., 0.2066, 0.2066, 0.2066],
        ...,
        [0.6868, 0.6868, 0.6868,  ..., 0.6868, 0.6868, 0.6868],
        [0.3769, 0.3769, 0.3769,  ..., 0.3769, 0.3769, 0.3769],
        [0.5489, 0.5489, 0.5489,  ..., 0.5489, 0.5489, 0.5489]],
       device='cuda:0')

## LR Schedular

### Train config

In [ ]:
peak_lr=2e-4
train_epoch=10
min_lr=2e-5
steps_per_epoch=len(train_dataloader)
total_iters=train_epoch*steps_per_epoch
warmup_frac=0.1
warmup_iters=int(total_iters*warmup_frac)
decay_frac=0.2
decay_iters=int(total_iters*(1-decay_frac))
eval_iters=100 #len(val_dataloader)


In [ ]:
def get_lr(it):
  if it<warmup_iters:
    return peak_lr*it/warmup_iters
  if it>warmup_iters and it<decay_iters:
    return peak_lr
  if it>=total_iters or decay_iters>=total_iters:
    return min_lr

  iters_into_decay=it-decay_iters
  decay_duration=total_iters-decay_iters
  decay_ratio=iters_into_decay/decay_duration
  decay_ratio=max(0.0, min(1.0, decay_ratio))
  return peak_lr-(peak_lr-min_lr)*decay_ratio



## Training loop

In [ ]:
def train():
  print("INITIALISING MODEL......")
  model=LlaDA(
      embed_dims=ModelArgs.embed_dims,
      num_heads=ModelArgs.num_heads,
      attn_dropout=ModelArgs.attn_dropout,
      num_blocks=ModelArgs.num_blocks,
      device=ModelArgs.device
  )
  model.to(ModelArgs.device)
  optimizer=torch.optim.AdamW(
      model.parameters(),
      lr=peak_lr,
      betas=(ModelArgs.beta1, ModelArgs.beta2),
      weight_decay=ModelArgs.AdamW_weight_decay
  )
  @torch.no_grad()
  def validate(model, val_dataloader):
    print("validating")
    model.eval()
    losses=torch.zeros(eval_iters, device=ModelArgs.device)
    for k, val_data in enumerate(val_dataloader):
      print(f"iter{k}")
      val_data=val_data.to(ModelArgs.device)
      if k>=eval_iters:
        break
      mc_loss=torch.zeros(16, device=ModelArgs.device)
      for i in range(16):
        input_ids=val_data[:, 0:ModelArgs.max_seq_len].contiguous()
        noisy_input_ids, mask_indices, p_mask=forward_process(input_ids)
        logits=model(noisy_input_ids)
        loss=F.cross_entropy(logits[mask_indices], input_ids[mask_indices], reduction='none')
        loss=loss.sum()/(input_ids.shape[0]*input_ids.shape[1])
        mc_loss[i]=loss
        print(loss)
      losses[k]=mc_loss.mean().item()
    out=losses.mean()
    model.train()
    return out

  loss_func=nn.CrossEntropyLoss(reduction=None)
  model.train()

  global_step=0
  best_val_loss=float("inf")
  for epoch in range(train_epoch):
    print(f"epoch: {epoch}")
    epoch_loss=0.0
    num_batches=0
    start_time=time.time()
    #if epoch%10==0 and epoch!=0:
    val_loss=validate(model, val_dataloader)
    print(f"val_loss: {val_loss}")
    if val_loss<best_val_loss:
        best_val_loss=val_loss
        torch.save(model.state_dict(), "best_model.pt")

    pbar=tqdm.tqdm(train_dataloader, desc=f"Training Epoch:{epoch+1}/{train_epoch}")
    for batch_idx, batch in enumerate(pbar):
      batch=batch.to(ModelArgs.device)
      lr=get_lr(global_step)
      for param_group in optimizer.param_groups:
        param_group['lr']=lr
      input_ids=batch[:, 0:ModelArgs.max_seq_len].contiguous()
      noisy_input_ids, mask_indices, p_mask=forward_process(input_ids)
      logits=model(noisy_input_ids)
      loss=F.cross_entropy(logits[mask_indices], input_ids[mask_indices], reduction='none')
      loss=loss.sum()/(input_ids.shape[0]*input_ids.shape[1])
      optimizer.zero_grad()
      loss.backward()
      torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
      optimizer.step()
      epoch_loss+=loss.item()
      num_batches+=1
      global_step+=1
      pbar.set_postfix({'loss': loss.item(),
                        'lr': lr,
                        'epoch_loss': epoch_loss/num_batches,
                        'time': time.time()-start_time
                        })
  print("\n" + "="*60)
  print("Training Complete! Running final validation...")
  final_val_loss = validate(model, val_dataloader)
  print(f"Final Validation Loss: {final_val_loss:.4f}")
  print(f"Best Validation Loss: {best_val_loss:.4f}")

  torch.save({
      'epoch': train_epoch,
      'model_state_dict': model.state_dict(),
      'optimizer_state_dict': optimizer.state_dict(),
      'val_loss': final_val_loss,
      'best_val_loss': best_val_loss,
      'global_step': global_step
  }, "final_model.pt")
  print("Saved final model to 'final_model.pt'")
  print("="*60)

  return model

## Run train

In [ ]:
# trained_model = train() #Ran on Kaggle Notebook

## OPTIMIZED Inference

In [ ]:
import torch
import torch.nn.functional as F
import sentencepiece as spm
import numpy as np
import time
from typing import Dict, Optional


def add_gumbel_noise(logits, temperature):
    """
    The Gumbel max is a method for sampling categorical distributions.
    According to arXiv:2409.02908, for MDM, low-precision Gumbel Max improves
    perplexity score but reduces generation quality. Thus, we use float64.

    Args:
        logits: Model logits (B, L, V)
        temperature: Gumbel temperature (0 = argmax, higher = more random)
    """
    if temperature == 0:
        return logits
    logits = logits.to(torch.float64)
    noise = torch.rand_like(logits, dtype=torch.float64)
    gumbel_noise = (-torch.log(noise)) ** temperature
    return logits.exp() / gumbel_noise


def get_num_transfer_tokens(mask_index, steps):
    """
    LLaDA employs a linear noise schedule. This function precomputes
    the number of tokens that need to be transitioned at each step.

    Args:
        mask_index: Boolean tensor indicating masked positions (B, L)
        steps: Number of denoising steps

    Returns:
        num_transfer_tokens: Number of tokens to unmask per step (B, steps)
    """
    mask_num = mask_index.sum(dim=1, keepdim=True)  # (B, 1)

    base = mask_num // steps
    remainder = mask_num % steps

    num_transfer_tokens = torch.zeros(
        mask_num.size(0), steps,
        device=mask_index.device,
        dtype=torch.int64
    ) + base

    # Distribute remainder across first few steps
    for i in range(mask_num.size(0)):
        num_transfer_tokens[i, :remainder[i]] += 1

    return num_transfer_tokens


@torch.no_grad()
def generate(model, prompt_ids, tokenizer, steps=64, gen_length=128,
             block_length=128, temperature=1.0, cfg_scale=0.,
             remasking='low_confidence', mask_id=6,
             logits_eos_inf=False, confidence_eos_eot_inf=False,
             return_timing=False):
    """
    Official LLaDA generation algorithm adapted for your model.

    Args:
        model: Your trained LlaDA model
        prompt_ids: List of token IDs for the prompt (or empty list)
        tokenizer: SentencePiece tokenizer
        steps: Sampling steps (e.g., 64, 128)
        gen_length: Number of tokens to generate
        block_length: Block length for semi-autoregressive generation
        temperature: Gumbel temperature (0=greedy, 1.0=default)
        cfg_scale: Classifier-free guidance scale (0=disabled)
        remasking: 'low_confidence' or 'random'
        mask_id: Your [MASK] token ID (default 6)
        logits_eos_inf: Set EOS logits to -inf
        confidence_eos_eot_inf: Set EOS confidence to -inf
        return_timing: If True, return (output, timing_dict)

    Returns:
        Generated token IDs (or tuple of (output, timing_dict) if return_timing=True)
    """
    timing = {
        'total': 0.0,
        'model_forward': 0.0,
        'sampling': 0.0,
        'remasking': 0.0,
        'per_step': []
    }

    start_time = time.time()

    device = next(model.parameters()).device

    # Convert prompt to tensor
    if len(prompt_ids) > 0:
        prompt = torch.tensor([prompt_ids], dtype=torch.long, device=device)
    else:
        prompt = torch.tensor([[]], dtype=torch.long, device=device)

    # Create sequence: prompt + masked tokens
    x = torch.full(
        (prompt.shape[0], prompt.shape[1] + gen_length),
        mask_id,
        dtype=torch.long,
        device=device
    )

    if prompt.shape[1] > 0:
        x[:, :prompt.shape[1]] = prompt.clone()

    # Track which positions are from prompt (shouldn't be modified)
    prompt_index = (x != mask_id)

    # Validate block configuration
    assert gen_length % block_length == 0, \
        f"gen_length ({gen_length}) must be divisible by block_length ({block_length})"
    num_blocks = gen_length // block_length

    assert steps % num_blocks == 0, \
        f"steps ({steps}) must be divisible by num_blocks ({num_blocks})"
    steps_per_block = steps // num_blocks

    # Generate block by block (semi-autoregressive)
    for num_block in range(num_blocks):
        block_start = prompt.shape[1] + num_block * block_length
        block_end = prompt.shape[1] + (num_block + 1) * block_length

        # Get mask indices for current block
        block_mask_index = (x[:, block_start:block_end] == mask_id)

        # Calculate how many tokens to unmask per step
        num_transfer_tokens = get_num_transfer_tokens(block_mask_index, steps_per_block)

        # Iterative denoising for this block
        for i in range(steps_per_block):
            step_start = time.time()

            mask_index = (x == mask_id)

            # Model forward pass
            forward_start = time.time()
            if cfg_scale > 0.:
                # Unconditional version (mask the prompt)
                un_x = x.clone()
                un_x[prompt_index] = mask_id
                x_ = torch.cat([x, un_x], dim=0)

                # Forward pass for both conditional and unconditional
                logits = model(x_)
                logits, un_logits = torch.chunk(logits, 2, dim=0)

                # Apply guidance
                logits = un_logits + (cfg_scale + 1) * (logits - un_logits)
            else:
                # Standard forward pass
                logits = model(x)

            if device.type == 'cuda':
                torch.cuda.synchronize()
            timing['model_forward'] += time.time() - forward_start

            # Sampling
            sampling_start = time.time()

            # Optional: Prevent EOS generation
            if logits_eos_inf:
                eos_id = tokenizer.piece_to_id('[EOS]')
                if eos_id >= 0:
                    logits[:, :, eos_id] = -torch.inf

            # Add Gumbel noise for sampling
            logits_with_noise = add_gumbel_noise(logits, temperature=temperature)

            # Sample next tokens (x0 prediction)
            x0 = torch.argmax(logits_with_noise, dim=-1)  # (B, L)

            if device.type == 'cuda':
                torch.cuda.synchronize()
            timing['sampling'] += time.time() - sampling_start

            # Remasking decision
            remasking_start = time.time()

            # Calculate confidence for remasking decision
            if remasking == 'low_confidence':
                # Use model confidence (softmax probability of predicted token)
                p = F.softmax(logits, dim=-1)

                # Optional: Set EOS/EoT confidence to -inf
                if confidence_eos_eot_inf:
                    eos_id = tokenizer.piece_to_id('[EOS]')
                    if eos_id >= 0:
                        p[:, :, eos_id] = 0

                # Get confidence of predicted tokens
                x0_p = torch.squeeze(
                    torch.gather(p, dim=-1, index=torch.unsqueeze(x0, -1)), -1
                )  # (B, L)

            elif remasking == 'random':
                # Random confidence (uniform sampling)
                x0_p = torch.rand((x0.shape[0], x0.shape[1]), device=x0.device)
            else:
                raise NotImplementedError(f"Remasking strategy '{remasking}' not implemented")

            # Don't unmask tokens beyond current block
            x0_p[:, block_end:] = -np.inf

            # Only consider currently masked positions
            x0 = torch.where(mask_index, x0, x)
            confidence = torch.where(mask_index, x0_p, -np.inf)

            # Select top-k most confident predictions to unmask
            transfer_index = torch.zeros_like(x0, dtype=torch.bool, device=x0.device)
            for j in range(confidence.shape[0]):
                if num_transfer_tokens[j, i] > 0:
                    _, select_index = torch.topk(
                        confidence[j],
                        k=num_transfer_tokens[j, i]
                    )
                    transfer_index[j, select_index] = True

            # Unmask selected positions
            x[transfer_index] = x0[transfer_index]

            if device.type == 'cuda':
                torch.cuda.synchronize()
            timing['remasking'] += time.time() - remasking_start

            step_time = time.time() - step_start
            timing['per_step'].append(step_time)

    timing['total'] = time.time() - start_time

    if return_timing:
        return x, timing
    return x


class LLaDAInference:
    """Wrapper class for easy inference"""

    def __init__(self, model, tokenizer_path, device='cuda'):
        """
        Initialize LLaDA inference

        Args:
            model: Trained LlaDA model
            tokenizer_path: Path to SentencePiece tokenizer
            device: Device to run on
        """
        if isinstance(device, str):
            device = torch.device(device)

        self.device = device
        self.model = model.to(self.device)
        self.model.eval()
        self.sp = spm.SentencePieceProcessor(model_file=tokenizer_path)
        self.mask_id = self.sp.piece_to_id('[MASK]')

        print(f"Initialized LLaDA Inference")
        print(f"  Mask token ID: {self.mask_id}")
        print(f"  Vocab size: {self.sp.get_piece_size()}")

    def generate_text(self, prompt="", max_length=256, steps=64,
                     temperature=1.0, cfg_scale=0.,
                     remasking='low_confidence', block_length=None,
                     verbose=True):
        """
        Generate text from prompt

        Args:
            prompt: Starting text (can be empty)
            max_length: Total length to generate
            steps: Number of denoising steps (more = better quality)
            temperature: Sampling temperature (0=greedy, 1.0=default, higher=more random)
            cfg_scale: Classifier-free guidance (0=disabled, 1.0-3.0=typical range)
            remasking: 'low_confidence' (default) or 'random'
            block_length: Block size for semi-autoregressive (None=same as max_length)
            verbose: Print timing information

        Returns:
            Generated text string (or tuple of (text, timing_dict) if verbose=True)
        """
        overall_start = time.time()

        # Encode prompt
        encode_start = time.time()
        if prompt:
            prompt_ids = self.sp.encode_as_ids(prompt)
        else:
            prompt_ids = []
        encode_time = time.time() - encode_start

        # Set block length
        if block_length is None:
            block_length = max_length

        # Generate with timing
        output_ids, timing = generate(
            model=self.model,
            prompt_ids=prompt_ids,
            tokenizer=self.sp,
            steps=steps,
            gen_length=max_length,
            block_length=block_length,
            temperature=temperature,
            cfg_scale=cfg_scale,
            remasking=remasking,
            mask_id=self.mask_id,
            logits_eos_inf=False,
            confidence_eos_eot_inf=False,
            return_timing=True
        )

        # Decode
        decode_start = time.time()
        output_ids = output_ids[0].cpu().tolist()

        # Filter out special tokens
        filtered_ids = [
            tid for tid in output_ids
            if tid not in [self.mask_id, 0, 1, 2, 3]
        ]

        generated_text = self.sp.decode(filtered_ids)
        decode_time = time.time() - decode_start

        # Add encoding/decoding time to timing dict
        timing['encoding'] = encode_time
        timing['decoding'] = decode_time
        timing['overall'] = time.time() - overall_start

        # Calculate statistics
        num_tokens = len(filtered_ids) - len(prompt_ids)
        timing['tokens_per_second'] = num_tokens / timing['total'] if timing['total'] > 0 else 0
        timing['ms_per_token'] = (timing['total'] * 1000) / num_tokens if num_tokens > 0 else 0
        timing['num_tokens'] = num_tokens
        timing['num_steps'] = len(timing['per_step'])
        timing['avg_step_time'] = np.mean(timing['per_step']) if timing['per_step'] else 0

        if verbose:
            self._print_timing(timing, prompt, generated_text)

        return generated_text, timing

    def _print_timing(self, timing: Dict, prompt: str, generated_text: str):
        """Print formatted timing information"""
        print("\n" + "="*80)
        print("GENERATION TIMING REPORT")
        print("="*80)

        if prompt:
            print(f"Prompt: {prompt[:50]}{'...' if len(prompt) > 50 else ''}")
        print(f"Generated tokens: {timing['num_tokens']}")
        print(f"Denoising steps: {timing['num_steps']}")
        print()

        print("TIMING BREAKDOWN:")
        print(f"  Encoding:       {timing['encoding']*1000:>8.2f} ms")
        print(f"  Generation:     {timing['total']*1000:>8.2f} ms")
        print(f"    - Forward:    {timing['model_forward']*1000:>8.2f} ms ({timing['model_forward']/timing['total']*100:.1f}%)")
        print(f"    - Sampling:   {timing['sampling']*1000:>8.2f} ms ({timing['sampling']/timing['total']*100:.1f}%)")
        print(f"    - Remasking:  {timing['remasking']*1000:>8.2f} ms ({timing['remasking']/timing['total']*100:.1f}%)")
        print(f"  Decoding:       {timing['decoding']*1000:>8.2f} ms")
        print(f"  Total:          {timing['overall']*1000:>8.2f} ms")
        print()

        print("PERFORMANCE:")
        print(f"  Tokens/second:  {timing['tokens_per_second']:>8.2f} tok/s")
        print(f"  Ms/token:       {timing['ms_per_token']:>8.2f} ms/tok")
        print(f"  Avg step time:  {timing['avg_step_time']*1000:>8.2f} ms/step")
        print("="*80)
        print()

    def infill(self, text_before, text_after, infill_length=20, steps=32,
               temperature=1.0, verbose=True):
        """
        Fill in text between two segments

        Args:
            text_before: Text before the blank
            text_after: Text after the blank
            infill_length: Number of tokens to generate in the middle
            steps: Number of denoising steps
            temperature: Sampling temperature
            verbose: Print timing information

        Returns:
            Complete text with infill (or tuple of (text, timing_dict) if verbose=True)
        """
        timing = {
            'total': 0.0,
            'model_forward': 0.0,
            'sampling': 0.0,
            'remasking': 0.0,
            'per_step': []
        }

        start_time = time.time()

        # Encode both parts
        before_ids = self.sp.encode_as_ids(text_before)
        after_ids = self.sp.encode_as_ids(text_after)

        # Create sequence: before + masks + after
        total_length = len(before_ids) + infill_length + len(after_ids)
        x = torch.full((1, total_length), self.mask_id, dtype=torch.long, device=self.device)

        # Fill in known parts
        x[0, :len(before_ids)] = torch.tensor(before_ids, device=self.device)
        x[0, -len(after_ids):] = torch.tensor(after_ids, device=self.device)

        # Only denoise the middle part
        prompt_index = (x != self.mask_id)
        mask_index = (x == self.mask_id)

        num_transfer_tokens = get_num_transfer_tokens(mask_index, steps)

        for i in range(steps):
            step_start = time.time()
            current_mask = (x == self.mask_id)

            # Forward pass
            forward_start = time.time()
            logits = self.model(x)
            if self.device.type == 'cuda':
                torch.cuda.synchronize()
            timing['model_forward'] += time.time() - forward_start

            # Sample
            sampling_start = time.time()
            logits_with_noise = add_gumbel_noise(logits, temperature)
            x0 = torch.argmax(logits_with_noise, dim=-1)
            if self.device.type == 'cuda':
                torch.cuda.synchronize()
            timing['sampling'] += time.time() - sampling_start

            # Calculate confidence
            remasking_start = time.time()
            p = F.softmax(logits, dim=-1)
            x0_p = torch.squeeze(
                torch.gather(p, dim=-1, index=torch.unsqueeze(x0, -1)), -1
            )

            x0 = torch.where(current_mask, x0, x)
            confidence = torch.where(current_mask, x0_p, -np.inf)

            # Select top-k to unmask
            transfer_index = torch.zeros_like(x0, dtype=torch.bool)
            _, select_index = torch.topk(confidence[0], k=num_transfer_tokens[0, i])
            transfer_index[0, select_index] = True

            x[transfer_index] = x0[transfer_index]

            if self.device.type == 'cuda':
                torch.cuda.synchronize()
            timing['remasking'] += time.time() - remasking_start

            step_time = time.time() - step_start
            timing['per_step'].append(step_time)

        timing['total'] = time.time() - start_time

        # Decode
        output_ids = x[0].cpu().tolist()
        filtered_ids = [tid for tid in output_ids if tid not in [self.mask_id, 0, 1, 2, 3]]
        generated_text = self.sp.decode(filtered_ids)

        # Calculate stats
        timing['num_tokens'] = infill_length
        timing['num_steps'] = len(timing['per_step'])
        timing['avg_step_time'] = np.mean(timing['per_step']) if timing['per_step'] else 0
        timing['tokens_per_second'] = infill_length / timing['total'] if timing['total'] > 0 else 0
        timing['ms_per_token'] = (timing['total'] * 1000) / infill_length if infill_length > 0 else 0

        if verbose:
            print("\n" + "="*80)
            print("INFILLING TIMING REPORT")
            print("="*80)
            print(f"Infilled tokens: {infill_length}")
            print(f"Denoising steps: {timing['num_steps']}")
            print()
            print("TIMING:")
            print(f"  Total:          {timing['total']*1000:>8.2f} ms")
            print(f"  Tokens/second:  {timing['tokens_per_second']:>8.2f} tok/s")
            print(f"  Ms/token:       {timing['ms_per_token']:>8.2f} ms/tok")
            print("="*80)
            print()

        return generated_text, timing


# =============================================================================
# USAGE EXAMPLES
# =============================================================================

def example_usage():
    """Example of how to use the official inference"""

    # Load model
    model = LlaDA(device='cuda')
    checkpoint = torch.load('/content/best_model_2.pt', map_location='cuda')
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
    else:
        model.load_state_dict(checkpoint)
    model.eval()

    # Initialize inference
    inferencer = LLaDAInference(model, 'tokenizer.model', device='cuda')

    print("="*80)
    print("EXAMPLE 1: Unconditional Generation")
    print("="*80)
    text = inferencer.generate_text(
        prompt="",
        max_length=128,
        steps=64,
        temperature=1.0
    )
    print(text)

    print("\n" + "="*80)
    print("EXAMPLE 2: Conditional Generation (Continue Prompt)")
    print("="*80)
    text = inferencer.generate_text(
        prompt="ROMEO: ",
        max_length=128,
        steps=64,
        temperature=0.9
    )
    print(text)

    print("\n" + "="*80)
    print("EXAMPLE 3: Semi-Autoregressive (Faster)")
    print("="*80)
    text = inferencer.generate_text(
        prompt="To be or not to be",
        max_length=128,
        steps=32,
        block_length=32,  # Generate in 32-token blocks
        temperature=1.0
    )
    print(text)

    print("\n" + "="*80)
    print("EXAMPLE 4: Infilling")
    print("="*80)
    text = inferencer.infill(
        text_before="ROMEO: ",
        text_after=" is the question.",
        infill_length=10,
        steps=32
    )
    print(text)

    print("\n" + "="*80)
    print("EXAMPLE 5: Greedy Decoding (temperature=0)")
    print("="*80)
    text = inferencer.generate_text(
        prompt="Once upon a time",
        max_length=64,
        steps=32,
        temperature=0  # Greedy
    )
    print(text)


if __name__ == "__main__":
    example_usage()

Initialized LLaDA Inference
  Mask token ID: 6
  Vocab size: 33488
EXAMPLE 1: Unconditional Generation

GENERATION TIMING REPORT
Generated tokens: 128
Denoising steps: 64

TIMING BREAKDOWN:
  Encoding:           0.00 ms
  Generation:      1296.04 ms
    - Forward:      953.51 ms (73.6%)
    - Sampling:     291.55 ms (22.5%)
    - Remasking:     48.02 ms (3.7%)
  Decoding:           0.60 ms
  Total:           1296.72 ms

PERFORMANCE:
  Tokens/second:     98.76 tok/s
  Ms/token:          10.13 ms/tok
  Avg step time:     20.24 ms/step

(', sir, I will not live, Pompey. POMPEY: Truly, good sir, I will not live, Pompey. POMPEY: Truly, sir, I will not live, Pompey, Pompey. POMPEY: Truly, sir, Pompey, Pompey; you will live you live, Pompey? POMPEY: Truly, sir, sir, I will not live, Pompey. POMPEY: Truly, good bawd, sir. POMPEY: I beseech you live, sir, I will not live,, Pompey. POMPEY: LUCIO: Well, sir, Pompey? POMPEY: Do live, sir. POMPEY: I beseech you live, sir, I will not bail', {'total'